# Assignment 1: Training models

## OLS Regression

This notebook fits and interprets a simple ordinary least squares (OLS) regression model for advertising budget and sales.

**Workflow**

1. Load and validate the data.
2. Derive and compute the OLS coefficients.
3. Interpret the fitted line and predict sales for a USD 12,000 budget.
4. Evaluate goodness of fit with R-squared.

## Problem setup

The variables use the following units:

- **Advertising Budget (X):** thousands of dollars.
- **Sales (Y):** thousands of units sold.

The model has the form `y_hat = b0 + b1 * x`, where `b0` is the intercept and `b1` is the slope.

In [1]:
from collections.abc import Sequence
from math import fsum, isfinite, isclose

In [2]:
months = tuple(range(1, 11))
advertising_budget = (3.0, 5.0, 2.0, 7.0, 8.0, 1.0, 4.0, 6.0, 9.0, 10.0)
sales = (6.0, 9.0, 4.0, 10.0, 12.0, 3.0, 7.0, 8.0, 13.0, 15.0)

# Validate the input before fitting the model.
if not (len(months) == len(advertising_budget) == len(sales)):
    raise ValueError("months, advertising_budget, and sales must have equal lengths")
if not all(isfinite(value) for value in (*advertising_budget, *sales)):
    raise ValueError("all observations must be finite numbers")

print(f"Observations: {len(months)}")
print(
    f"Advertising budget range: {min(advertising_budget):.1f} to {max(advertising_budget):.1f} thousand dollars"
)
print(f"Sales range: {min(sales):.1f} to {max(sales):.1f} thousand units")

Observations: 10
Advertising budget range: 1.0 to 10.0 thousand dollars
Sales range: 3.0 to 15.0 thousand units


## Data

| Month | Advertising Budget (X) | Sales (Y) |
|---:|---:|---:|
| 1 | 3.0 | 6.0 |
| 2 | 5.0 | 9.0 |
| 3 | 2.0 | 4.0 |
| 4 | 7.0 | 10.0 |
| 5 | 8.0 | 12.0 |
| 6 | 1.0 | 3.0 |
| 7 | 4.0 | 7.0 |
| 8 | 6.0 | 8.0 |
| 9 | 9.0 | 13.0 |
| 10 | 10.0 | 15.0 |

## OLS method

For a simple linear model, calculate the coefficients with:

- `b1 = Sxy / Sxx`
- `b0 = y_mean - b1 * x_mean`
- `Sxy = sum((x_i - x_mean) * (y_i - y_mean))`
- `Sxx = sum((x_i - x_mean) ** 2)`

The implementation below uses `math.fsum` for accurate floating-point summation and checks invalid inputs.

In [3]:
def fit_ols(x: Sequence[float], y: Sequence[float]) -> dict[str, float]:
    """Fit y = b0 + b1*x and return coefficients and key sums."""

    if len(x) != len(y):
        raise ValueError("x and y must have equal lengths")
    if len(x) < 2:
        raise ValueError("at least two observations are required")
    if not all(isfinite(value) for value in (*x, *y)):
        raise ValueError("x and y must contain only finite values")

    x_mean = fsum(x) / len(x)
    y_mean = fsum(y) / len(y)
    sxx = fsum((value - x_mean) ** 2 for value in x)
    if sxx == 0:
        raise ValueError("the predictor must vary")
    sxy = fsum(
        (x_value - x_mean) * (y_value - y_mean) for x_value, y_value in zip(x, y)
    )

    slope = sxy / sxx
    intercept = y_mean - slope * x_mean
    return {
        "intercept": intercept,
        "slope": slope,
        "x_mean": x_mean,
        "y_mean": y_mean,
        "sxx": sxx,
        "sxy": sxy,
    }

In [4]:
model = fit_ols(advertising_budget, sales)

print("Regression coefficients")
print(f"  Intercept (b0): {model['intercept']:.6f}")
print(f"  Slope (b1):     {model['slope']:.6f}")
print()
print("Fitted line")
print(f"  y_hat = {model['intercept']:.4f} + {model['slope']:.4f}x")

# Hand-calculated values provide a simple implementation check.
assert isclose(model["intercept"], 1.7333333333, rel_tol=1e-10)
assert isclose(model["slope"], 1.2666666667, rel_tol=1e-10)

Regression coefficients
  Intercept (b0): 1.733333
  Slope (b1):     1.266667

Fitted line
  y_hat = 1.7333 + 1.2667x


## Interpretation of the coefficients

- **Slope:** The model estimates an increase of about **1.2667 thousand units sold** for each additional **1.0 thousand dollars** spent on advertising. This is an association in this sample, not proof that advertising alone causes the increase.
- **Intercept:** At an advertising budget of zero, the model predicts about **1.7333 thousand units sold**. Because zero is outside the observed budget range, this interpretation is an extrapolation and should be treated cautiously.

In [5]:
def predict(model: dict[str, float], x: float) -> float:
    """Predict sales in thousands of units for a budget in thousands of dollars."""

    return model["intercept"] + model["slope"] * x


budget_usd = 12_000
budget_thousands = budget_usd / 1_000
predicted_sales_thousands = predict(model, budget_thousands)

print(
    f"Advertising budget: ${budget_usd:,.0f} ({budget_thousands:.1f} thousand dollars)"
)
print(f"Predicted sales: {predicted_sales_thousands:.4f} thousand units")
print(f"Approximately {predicted_sales_thousands * 1_000:,.0f} units")

assert isclose(predicted_sales_thousands, 16.9333333333, rel_tol=1e-10)

Advertising budget: $12,000 (12.0 thousand dollars)
Predicted sales: 16.9333 thousand units
Approximately 16,933 units


## Prediction for a USD 12,000 budget

USD 12,000 equals `x = 12.0` because the predictor is measured in thousands of dollars. Substituting into the fitted line gives:

`y_hat = 1.7333 + 1.2667(12.0) = 16.9333` thousand units.

The predicted sales are therefore approximately **16.9333 thousand units**, or **16,933 units**. The observed budgets range from 1.0 to 10.0, so this prediction is an extrapolation beyond the available data.

## Model evaluation with R-squared

Use the fitted values to calculate:

- `SSE = sum((y_i - y_hat_i) ** 2)`
- `SST = sum((y_i - y_mean) ** 2)`
- `R^2 = 1 - SSE / SST`

`R^2` measures the proportion of the variation in observed sales explained by the fitted linear relationship.

In [6]:
def evaluate_r_squared(
    actual: Sequence[float], predicted: Sequence[float]
) -> dict[str, float]:
    """Return SSE, SST, and R-squared for a set of predictions."""

    if len(actual) != len(predicted):
        raise ValueError("actual and predicted must have equal lengths")
    if not actual:
        raise ValueError("at least one observation is required")

    mean_actual = fsum(actual) / len(actual)
    sse = fsum(
        (actual_value - predicted_value) ** 2
        for actual_value, predicted_value in zip(actual, predicted)
    )
    sst = fsum((actual_value - mean_actual) ** 2 for actual_value in actual)
    if sst == 0:
        raise ValueError("R-squared is undefined when actual values do not vary")

    return {"sse": sse, "sst": sst, "r_squared": 1 - sse / sst}

In [7]:
fitted_sales = tuple(predict(model, budget) for budget in advertising_budget)
evaluation = evaluate_r_squared(sales, fitted_sales)

print(f"SSE: {evaluation['sse']:.6f}")
print(f"SST: {evaluation['sst']:.6f}")
print(f"R-squared: {evaluation['r_squared']:.6f}")
print(f"Explained variation: {evaluation['r_squared'] * 100:.2f}%")

assert isclose(evaluation["sse"], 3.7333333333, rel_tol=1e-10)
assert isclose(evaluation["sst"], 136.1, rel_tol=1e-10)
assert isclose(evaluation["r_squared"], 0.9725691893, rel_tol=1e-10)

SSE: 3.733333
SST: 136.100000
R-squared: 0.972569
Explained variation: 97.26%


## Interpretation of R-squared

The model has `R^2 = 0.972569`, so approximately **97.26% of the variation in observed sales** is explained by advertising budget in this sample. The remaining variation is represented by the model residuals and may be related to other factors or random variation.

A high R-squared does not establish causation, guarantee future accuracy, or remove the risk of extrapolating beyond the observed budget range.

## Final answers

- **Fitted equation:** `Sales_hat = 1.7333 + 1.2667 * Advertising Budget`
- **Intercept:** `1.7333` thousand units sold when the advertising budget is zero.
- **Slope:** `1.2667` thousand additional units sold per additional thousand dollars of advertising budget.
- **Prediction at USD 12,000:** `16.9333` thousand units, approximately `16,933` units.
- **R-squared:** `0.972569`, meaning approximately `97.26%` of the sample variation in sales is explained by the model.